In [1]:
import xarray as xr
import pandas as pd
import hyperparameters as hp
from pyproj import Transformer

In [4]:
met_path = r"C:\Users\ameli\Documents\Uni\year-3-notes\diss\Dataset\rainfall_hadukgrid_uk_region_day_18910101-20241231.nc"
cur_path = r"C:\Users\ameli\Documents\Uni\year-3-notes\diss\Dataset\rainfall_hadukgrid_uk_60km_day_20220901-20220930.nc"
graph_path = hp.SUBSETGRAPH_PATH

pd.set_option('display.max_rows', 500)

In [10]:
def load_in_weather_data(path, region_idx=16):
    ds = xr.open_dataset(path, engine='netcdf4')

    start_date = '2020-12-31'
    end_date = '2024-09-29'

    ds_subset = ds.sel(region=region_idx, time=slice(start_date, end_date))

    df = ds_subset.to_dataframe().reset_index()


    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')
    df = df.drop(['region', 'geo_region', 'time_bnds', 'bnds'], axis=1)
    df = df.drop_duplicates()

    df = df.resample("15min").ffill()

    cut_off_time = df.index.min() + pd.Timedelta(hours=12)
    df = df[df.index >= cut_off_time]

    return df

weather_df = load_in_weather_data(met_path,)
#weather_df.head(100)

In [11]:
def find_time_gaps(df, freq="15min"):
    """
    Identify missing timestamp ranges and NaN sections in a time-indexed dataframe.
    Assumes the index is a DatetimeIndex.
    """

    df = df.sort_index()

    # Expected full timeline
    expected_index = pd.date_range(df.index.min(), df.index.max(), freq=freq)

    # Missing timestamps
    missing_times = expected_index.difference(df.index)

    gap_ranges = []
    if len(missing_times) > 0:
        groups = (missing_times.to_series().diff() != pd.Timedelta(freq)).cumsum()
        for _, g in missing_times.to_series().groupby(groups):
            gap_ranges.append({
                "start": g.iloc[0],
                "end": g.iloc[-1],
                "missing_points": len(g)
            })

    gaps_df = pd.DataFrame(gap_ranges)

    # Check NaN sections in data
    nan_summary = df.isna().sum()
    nan_cols = nan_summary[nan_summary > 0]

    return gaps_df, nan_cols

gaps, nan_cols = find_time_gaps(weather_df)

print("Missing time ranges:")
print(gaps)

print("\nColumns with NaNs:")
print(nan_cols)

Missing time ranges:
Empty DataFrame
Columns: []
Index: []

Columns with NaNs:
Series([], dtype: int64)


In [ ]:
def get_weather_samples(weather_df, flowdata_samples_df, sample_length=hp.TOTAL_WINDOW):

    weather_samples_df = pd.DataFrame(columns=hp.WEATHER_COLS)

    for sample in flowdata_samples_df.itertuples():
        sample_time = sample.time


        start_index = weather_df.index.get_loc(sample_time)
        end_index = start_index + sample_length

        weather_sample = {}

        for col in hp.WEATHER_COLS:
            weather_sample[col] = weather_df.iloc[start_index:end_index][col].values

        index = len(weather_samples_df)
        weather_samples_df.loc[index] = weather_sample

    return weather_samples_df



NameError: name 'find_time_gaps' is not defined

In [ ]:


ds = xr.open_dataset(cur_path, engine='netcdf4')

#print(ds)

df = ds.to_dataframe().reset_index()

print(df.head(1000))

#print(ds['longitude'].values)

                   time  projection_y_coordinate  projection_x_coordinate  \
0   2022-09-01 12:00:00                 -90000.0                -210000.0   
1   2022-09-01 12:00:00                 -90000.0                -210000.0   
2   2022-09-01 12:00:00                 -90000.0                -150000.0   
3   2022-09-01 12:00:00                 -90000.0                -150000.0   
4   2022-09-01 12:00:00                 -90000.0                 -90000.0   
..                  ...                      ...                      ...   
995 2022-09-02 12:00:00                 270000.0                  30000.0   
996 2022-09-02 12:00:00                 270000.0                  90000.0   
997 2022-09-02 12:00:00                 270000.0                  90000.0   
998 2022-09-02 12:00:00                 270000.0                 150000.0   
999 2022-09-02 12:00:00                 270000.0                 150000.0   

     bnds  rainfall  transverse_mercator           time_bnds  \
0       0  

In [19]:
dma_df = pd.read_csv(graph_path, index_col=0)
sensor_df = dma_df[dma_df['SensorIndicator'] == 1]

#print(sensor_df)

transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)


for index, row in sensor_df.iterrows():
    easting_start = row['Start_x']
    northing_start = row['Start_y']
    easting_end = row['End_x']
    northing_end = row['End_y']

    lat_start, lon_start = transformer.transform(easting_start, northing_start)
    lat_end, lon_end = transformer.transform(easting_end, northing_end)

    print(f"Sensor {row['SensorDMA']}:")
    print("Start Latitude/Longitude:", lat_start, lon_start)
    print("End Latitude/Longitude:", lat_end, lon_end)
    print('')




Sensor 1615.0:
Start Latitude/Longitude: -0.22290361267986755 53.60767115400114
End Latitude/Longitude: -0.22289531826435388 53.60766313802302

Sensor 919.0:
Start Latitude/Longitude: -0.21269855874350396 53.610231125367655
End Latitude/Longitude: -0.2126863988708681 53.61021480754154

Sensor 157.0:
Start Latitude/Longitude: -0.21679991139498564 53.619713668628904
End Latitude/Longitude: -0.21678673144375693 53.6197307770553

Sensor 1959.0:
Start Latitude/Longitude: -0.21770820550428144 53.6146794971418
End Latitude/Longitude: -0.217826640472438 53.61481087181967

Sensor 1016.0:
Start Latitude/Longitude: -0.1623429437304934 53.613261583400096
End Latitude/Longitude: -0.16228360980920056 53.61328552846485

Sensor 1994.0:
Start Latitude/Longitude: -0.20109200962196094 53.610806450264974
End Latitude/Longitude: -0.20107159638947164 53.61081689489075

Sensor 1870.0:
Start Latitude/Longitude: -0.20041297351899834 53.621150267000104
End Latitude/Longitude: -0.2004479555587517 53.621164286304

In [52]:
def load_in_dataset(path, region_idx):
    ds = xr.open_dataset(path, engine='netcdf4')

    start_date = '2020-01-01'
    end_date = '2024-09-29'

    #print("Available regions:", ds['geo_region'].values)
    #print("Available regions:", ds['region'].values)

    #print(ds)

    ds_subset = ds.sel(region=region_idx, time=slice(start_date, end_date))

    df = ds_subset.to_dataframe().reset_index()

    #print(df)

    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')
    df = df.drop(['region', 'geo_region', 'time_bnds', 'bnds'], axis=1)
    df = df.drop_duplicates()

    #print(df)

    return df

weather_df = load_in_dataset(met_path, 16)
weather_df = weather_df.resample("15min").ffill()
weather_df.head(100)


,rainfall
time,
2020-01-01 12:00:00,0.01804
2020-01-01 12:15:00,0.01804
2020-01-01 12:30:00,0.01804
2020-01-01 12:45:00,0.01804
2020-01-01 13:00:00,0.01804
...,...
2020-01-02 11:45:00,0.01804
2020-01-02 12:00:00,1.32300
2020-01-02 12:15:00,1.32300
